# 02 · EB, τ grid, stability → `thresholds`, `alerts`

In [ ]:
from pyspark.sql import functions as F, Window as W
from pyspark.sql.types import DoubleType

# Read isolates written by 01_ingest
iso = spark.table('isolates')

# Successes (resistant isolates) from %
iso = iso.withColumn('r', F.round(F.col('percent_resistant')/100.0 * F.col('n_tested')).cast('int'))
iso.cache(); iso.count()

# Simple Bayes–Laplace posterior (Beta): alpha=r+1, beta=n-r+1
eb = (iso
  .select('province','organism','antibiotic','specimen','year','n_tested','r')
  .withColumn('alpha', F.col('r') + F.lit(1.0))
  .withColumn('beta',  (F.col('n_tested') - F.col('r')) + F.lit(1.0))
  .withColumn('theta_hat', (F.col('alpha'))/(F.col('alpha')+F.col('beta')))
)

# UDF for Pr(theta > tau) where theta~Beta(alpha,beta). Uses scipy if available; fallback to mpmath.
def _pr_exceed(alpha, beta, tau):
    try:
        from scipy.special import betainc
        # CDF at tau is I_tau(alpha, beta); Pr > tau = 1 - CDF
        return float(1.0 - betainc(alpha, beta, tau))
    except Exception:
        try:
            import mpmath as mp
            return float(1.0 - mp.betainc(alpha, beta, 0, tau, regularized=True))
        except Exception:
            # Very rough fallback: compare posterior mean only
            return 1.0 if (alpha/(alpha+beta)) > tau else 0.0

pr_exceed_udf = F.udf(_pr_exceed, DoubleType())

# Build a tau grid 0.10..0.40 step 0.01
taus = [round(x/100, 2) for x in range(10, 41)]
tau_df = spark.createDataFrame([(t,) for t in taus], ['tau'])

# Cross-join to compute Pr(theta>tau) for all taus
eb_tau = (eb.crossJoin(tau_df)
  .withColumn('pr_exceed_tau', pr_exceed_udf(F.col('alpha'), F.col('beta'), F.col('tau')))
)

# Choose tau per (organism, antibiotic, specimen) using latest year: maximize mean pr_exceed_tau
latest_year = eb_tau.agg(F.max('year').alias('y')).collect()[0]['y']
crit = (eb_tau.where(F.col('year')==latest_year)
  .groupBy('organism','antibiotic','specimen','tau')
  .agg(F.avg('pr_exceed_tau').alias('score'))
)
w = W.partitionBy('organism','antibiotic','specimen').orderBy(F.col('score').desc(), F.col('tau').asc())
thresholds = (crit
  .withColumn('rank', F.row_number().over(w))
  .where('rank=1')
  .select('organism','antibiotic','specimen',F.lit(latest_year).alias('year_ref'),'tau', F.lit('grid_search_latest_year').alias('method'))
)

# Attach chosen tau to each row and compute gate/stability
alerts = (eb
  .join(thresholds.select('organism','antibiotic','specimen','tau'), ['organism','antibiotic','specimen'], 'left')
  .withColumn('pr_exceed_tau', pr_exceed_udf(F.col('alpha'), F.col('beta'), F.col('tau')))
  .withColumn('gate', (F.col('pr_exceed_tau') >= F.lit(0.80)))
)

# Persistence: gate true for current year and previous year
w_year = W.partitionBy('province','organism','antibiotic','specimen').orderBy('year')
alerts = alerts.withColumn('gate_prev', F.lag('gate').over(w_year))
alerts = alerts.withColumn('persistence', F.when(F.col('gate') & F.col('gate_prev'), F.lit(True)).otherwise(F.lit(False)))

# 3-year slope over pr_exceed_tau (approx: (y3 - y1)/2 when we have 3 points)
alerts = alerts.withColumn('pr_prev', F.lag('pr_exceed_tau',1).over(w_year))
alerts = alerts.withColumn('pr_prev2', F.lag('pr_exceed_tau',2).over(w_year))
alerts = alerts.withColumn('slope3', (F.col('pr_exceed_tau') - F.col('pr_prev2'))/F.lit(2.0))
alerts = alerts.withColumn('slope_ok', F.when(F.col('slope3') >= F.lit(0.05), F.lit(True)).otherwise(F.lit(False)))

# Final stability and impact
alerts = (alerts
  .withColumn('is_stable_alert', F.col('gate') & (F.col('persistence') | F.col('slope_ok')))
  .withColumn('impact_score', F.when(F.col('theta_hat') > F.col('tau'), (F.col('theta_hat') - F.col('tau'))*F.lit(1000.0)).otherwise(F.lit(0.0)))
  .withColumn('reason', F.when(F.col('is_stable_alert'), F.when(F.col('persistence'), F.lit('gate+persistence')).otherwise(F.lit('gate+trend'))).otherwise(F.lit('no_stable_rule')))
)

# Write Delta tables
spark.sql('DROP TABLE IF EXISTS thresholds')
thresholds.write.mode('overwrite').format('delta').saveAsTable('thresholds')
spark.sql('REFRESH TABLE thresholds')

spark.sql('DROP TABLE IF EXISTS alerts')
alerts.select('province','organism','antibiotic','specimen','year','n_tested','theta_hat','tau','pr_exceed_tau','is_stable_alert','impact_score','reason').\
  write.mode('overwrite').format('delta').saveAsTable('alerts')
spark.sql('REFRESH TABLE alerts')

display(spark.table('thresholds').orderBy('organism','antibiotic','specimen'))
display(spark.table('alerts').orderBy('province','organism','antibiotic','year'))
